In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Primitive-Eingaben und -Ausgaben

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Die Beta-Version eines neuen Ausführungsmodells ist jetzt verfügbar. Das gerichtete Ausführungsmodell bietet mehr Flexibilität bei der Anpassung deines Fehlerminderungsworkflows. Weitere Informationen findest du im Leitfaden zum [Gerichteten Ausführungsmodell](/guides/directed-execution-model).


<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese oder neuere Versionen zu verwenden.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Diese Seite gibt einen Überblick über die Eingaben und Ausgaben der Qiskit Runtime Primitives, die Workloads auf IBM Quantum&reg;-Rechenressourcen ausführen. Diese Primitives ermöglichen es dir, vektorisierte Workloads mithilfe einer Datenstruktur namens **Primitive Unified Bloc (PUB)** effizient zu definieren. Diese PUBs sind die grundlegende Arbeitseinheit, die ein QPU zur Ausführung dieser Workloads benötigt. Sie werden als Eingaben für die [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run)-Methode der Sampler- und Estimator-Primitives verwendet, die den definierten Workload als Job ausführen. Sobald der Job abgeschlossen ist, werden die Ergebnisse in einem Format zurückgegeben, das sowohl von den verwendeten PUBs als auch von den für die Sampler- oder Estimator-Primitives angegebenen Laufzeitoptionen abhängt.
<span id="pubs"></span>
## Überblick über PUBs
Beim Aufrufen der [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run)-Methode eines Primitive ist das Hauptargument eine `list` aus einem oder mehreren Tupeln – eines für jeden Circuit, der vom Primitive ausgeführt wird. Jedes dieser Tupel gilt als PUB, und die erforderlichen Elemente jedes Tupels in der Liste hängen vom verwendeten Primitive ab. Die in diesen Tupeln bereitgestellten Daten können auch in verschiedenen Formen angeordnet werden, um durch Broadcasting Flexibilität in einem Workload zu bieten – die entsprechenden Regeln sind in einem [nachfolgenden Abschnitt](#broadcasting-rules) beschrieben.

### Estimator-PUB
Beim Estimator-Primitive sollte das PUB-Format höchstens vier Werte enthalten:
- Einen einzelnen `QuantumCircuit`, der ein oder mehrere [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)-Objekte enthalten kann
- Eine Liste aus einem oder mehreren Observablen, die die zu schätzenden Erwartungswerte angeben und in einem Array angeordnet sind (zum Beispiel ein einzelnes Observable als 0-d-Array, eine Liste von Observablen als 1-d-Array usw.). Die Daten können in jedem der `ObservablesArrayLike`-Formate vorliegen, etwa `Pauli`, `SparsePauliOp`, `PauliList` oder `str`.
   > **Note:** Wenn du zwei kommutierende Observablen in verschiedenen PUBs, aber mit demselben Circuit hast, werden sie nicht mit derselben Messung geschätzt. Jeder PUB steht für eine andere Messbasis, daher sind für jeden PUB separate Messungen erforderlich. Um sicherzustellen, dass kommutierende Observablen mit derselben Messung geschätzt werden, müssen sie innerhalb desselben PUBs gruppiert werden.
- Eine Sammlung von Parameterwerten, gegen die der Circuit gebunden wird. Dies kann als einzelnes array-ähnliches Objekt angegeben werden, bei dem der letzte Index über die `Parameter`-Objekte des Circuits läuft, oder es kann weggelassen werden (bzw. äquivalent auf `None` gesetzt werden), wenn der Circuit keine `Parameter`-Objekte hat.
- (Optional) eine Zielpräzision für die zu schätzenden Erwartungswerte

### Sampler-PUB
Beim Sampler-Primitive enthält das PUB-Tupel höchstens drei Werte:
- Einen einzelnen `QuantumCircuit`, der ein oder mehrere [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter)-Objekte enthalten kann
   *Hinweis: Diese Circuits sollten außerdem Messanweisungen für jeden der zu messenden Qubits enthalten.*
- Eine Sammlung von Parameterwerten, gegen die der Circuit gebunden wird $\theta_k$ (nur erforderlich, wenn `Parameter`-Objekte verwendet werden, die zur Laufzeit gebunden werden müssen)
- (Optional) eine Anzahl von Shots, mit denen der Circuit gemessen werden soll
---

Der folgende Code zeigt ein Beispiel für vektorisierte Eingaben an das `Estimator`-Primitive und führt sie als einzelnes `RuntimeJobV2`-Objekt auf einem IBM&reg;-Backend aus.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV2 as Sampler,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

### Broadcasting-Regeln
Die PUBs fassen Elemente aus mehreren Arrays (Observablen und Parameterwerte) zusammen, indem sie dieselben Broadcasting-Regeln wie NumPy befolgen. Dieser Abschnitt fasst diese Regeln kurz zusammen. Eine ausführliche Erklärung findest du in der [NumPy-Dokumentation zu Broadcasting-Regeln](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Regeln:

* Eingabe-Arrays müssen nicht dieselbe Anzahl an Dimensionen haben.
  * Das resultierende Array hat die gleiche Anzahl an Dimensionen wie das Eingabe-Array mit der größten Dimension.
  * Die Größe jeder Dimension ist die größte Größe der entsprechenden Dimension.
  * Fehlende Dimensionen werden als Größe eins angenommen.
* Dimensionsvergleiche beginnen mit der äußersten rechten Dimension und setzen sich nach links fort.
* Zwei Dimensionen sind kompatibel, wenn ihre Größen gleich sind oder wenn eine davon 1 ist.

Beispiele für Array-Paare, die Broadcasting unterstützen:

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

Beispiele für Array-Paare, die kein Broadcasting unterstützen:

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

`EstimatorV2` gibt einen Erwartungswertes-Schätzwert für jedes Element der ausgebroadcasteten Form zurück.

Hier sind einige Beispiele für häufige Muster, ausgedrückt in Begriffen des Array-Broadcastings. Die zugehörigen visuellen Darstellungen sind in der folgenden Abbildung zu sehen:

Parameterwertmengen werden durch n×m-Arrays dargestellt, und Observable-Arrays werden durch eine oder mehrere einspaltige Arrays dargestellt. In jedem der vorherigen Code-Beispiele werden die Parameterwertmengen mit ihrem Observable-Array kombiniert, um die resultierenden Erwartungswertes-Schätzwerte zu erstellen.

 - *Beispiel 1*: (einzelnes Observable broadcasted) hat eine Parameterwertmenge als 5×1-Array und ein 1×1-Observable-Array. Das eine Element im Observable-Array wird mit jedem Element in der Parameterwertmenge kombiniert, um ein einzelnes 5×1-Array zu erstellen, bei dem jedes Element eine Kombination des ursprünglichen Elements aus der Parameterwertmenge mit dem Element aus dem Observable-Array ist.

 - *Beispiel 2*: (Zip) hat eine 5×1-Parameterwertmenge und ein 5×1-Observable-Array. Die Ausgabe ist ein 5×1-Array, bei dem jedes Element eine Kombination des n-ten Elements in der Parameterwertmenge mit dem n-ten Element im Observable-Array ist.

  - *Beispiel 3*: (Äußeres Produkt) hat eine 1×6-Parameterwertmenge und ein 4×1-Observable-Array. Ihre Kombination ergibt ein 4×6-Array, das durch die Kombination jedes Elements der Parameterwertmenge mit *jedem* Element im Observable-Array entsteht – daher wird jeder Parameterwert zu einer ganzen Spalte in der Ausgabe.

  - *Beispiel 4*: (Standard-nd-Verallgemeinerung) hat ein 3×6-Parameterwertmengen-Array und zwei 3×1-Observable-Arrays. Diese kombinieren sich zu zwei 3×6-Ausgabe-Arrays, ähnlich wie im vorherigen Beispiel.

![Dieses Bild zeigt mehrere visuelle Darstellungen des Array-Broadcastings](../docs/images/guides/primitive-input-output/broadcasting.svg "Visuelle Darstellung des Broadcastings")

In [4]:
# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=4096, num_bits=10>)

The shape of register `meas` is (4096, 2).

The bytes in register `alpha`, shot by shot:
[[  3 254]
 [  0   0]
 [  3 255]
 ...
 [  0   0]
 [  3 255]
 [  0   0]]



<Admonition type="tip" title="SparsePauliOp">
Jede `SparsePauliOp` zählt in diesem Kontext als ein einzelnes Element, unabhängig von der Anzahl der darin enthaltenen Paulis. Daher haben für die Zwecke dieser Broadcasting-Regeln alle folgenden Elemente dieselbe Form:

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'1111111110': 199, '0000000000': 1337, '1111111111': 1052, '1111111000': 33, '1110000000': 65, '1100100000': 2, '1100000000': 25, '0010001110': 1, '0000000011': 30, '1111111011': 58, '1111111010': 25, '0000000110': 7, '0010000001': 11, '0000000001': 179, '1110111110': 6, '1111110000': 33, '1111101111': 49, '1110111111': 40, '0000111010': 2, '0100000000': 35, '0000000010': 51, '0000100000': 31, '0110000000': 7, '0000001111': 22, '1111111100': 24, '1011111110': 5, '0001111111': 58, '0000111111': 24, '1111101110': 10, '0000010001': 5, '0000001001': 2, '0011111111': 38, '0000001000': 11, '1111100000': 34, '0111111111': 45, '0000000100': 18, '0000000101': 2, '1011111111': 11, '1110000001': 13, '1101111000': 1, '0010000000': 52, '0000010000': 17, '0000011111': 15, '1110100001': 1, '0111111110': 9, '0000000111': 19, '1101111111': 15, '1111110111': 17, '0011111110': 5, '0001101110': 1, '0111111011': 6, '0100001000': 2, '0010001111': 1, '1111011000': 1, '0000111110': 4, '0011110010': 1

Die folgenden Operatorlisten sind zwar inhaltlich äquivalent, haben aber unterschiedliche Formen:

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=4096, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=4096, num_bits=9>)


</Admonition>
## Überblick über Primitive-Ausgaben
Sobald ein oder mehrere PUBs zur Ausführung an ein QPU gesendet wurden und ein Job erfolgreich abgeschlossen ist, werden die Daten als [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)-Container-Objekt zurückgegeben, auf das durch Aufrufen der `RuntimeJobV2.result()`-Methode zugegriffen werden kann. Das `PrimitiveResult` enthält eine iterierbare Liste von [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult)-Objekten, die die Ausführungsergebnisse für jeden PUB enthalten. Je nach verwendetem Primitive sind diese Daten entweder Erwartungswerte und ihre Fehlerbalken (beim Estimator) oder Stichproben der Circuit-Ausgabe (beim Sampler).

Jedes Element dieser Liste entspricht einem PUB, der an die `run()`-Methode des Primitive übergeben wurde (ein Job, der zum Beispiel mit 20 PUBs gesendet wird, gibt ein `PrimitiveResult`-Objekt zurück, das eine Liste mit 20 `PubResults` enthält, eines für jeden PUB).

Jedes dieser `PubResult`-Objekte besitzt sowohl ein `data`- als auch ein `metadata`-Attribut. Das `data`-Attribut ist ein angepasstes [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin), das die eigentlichen Messwerte, Standardabweichungen usw. enthält. Dieses `DataBin` hat verschiedene Attribute, abhängig von der Form oder Struktur des zugehörigen PUBs sowie den vom verwendeten Primitive angegebenen Fehlerminderungsoptionen (zum Beispiel [ZNE](./error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) oder [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)). Das `metadata`-Attribut hingegen enthält Informationen über die verwendeten Laufzeit- und Fehlerminderungsoptionen (erläutert im Abschnitt [Ergebnis-Metadaten](#result-metadata) auf dieser Seite).

Im Folgenden ist eine visuelle Übersicht der `PrimitiveResult`-Datenstruktur dargestellt:

<Tabs>
    <TabItem value="estimator" label="Estimator-Ausgabe">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
    <TabItem value="sampler" label="Sampler-Ausgabe">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second pub
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
</Tabs>

Kurz gesagt: Ein einzelner Job gibt ein `PrimitiveResult`-Objekt zurück und enthält eine Liste aus einem oder mehreren `PubResult`-Objekten. Diese `PubResult`-Objekte speichern dann die Messdaten für jeden PUB, der dem Job übergeben wurde.

Jedes `PubResult` besitzt je nach dem für den Job verwendeten Primitive-Typ unterschiedliche Formate und Attribute. Die Details werden im Folgenden erläutert.

### Estimator-Ausgabe
Jedes `PubResult` des Estimator-Primitive enthält mindestens ein Array von Erwartungswerten (`PubResult.data.evs`) und zugehörigen Standardabweichungen (entweder `PubResult.data.stds` oder `PubResult.data.ensemble_standard_error`, abhängig vom verwendeten `resilience_level`), kann aber je nach angegebenen Fehlerminderungsoptionen weitere Daten enthalten.

Der folgende Code-Ausschnitt beschreibt das `PrimitiveResult`- (und zugehörige `PubResult`-)Format für den oben erstellten Job.

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (4096, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (4096, 2).
The bytes in register `beta`, shot by shot:
[[  0 135]
 [  0 247]
 [  1 247]
 ...
 [  0   0]
 [  1 224]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (4096, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  0 135]
 [  0 247]
 [  1 247]
 [  1 128]
 [  1 255]]

Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: 0.068359375
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: 0.06396484375

The shape of the merged results is (4096, 2).
The bytes of the merged results:
[[  1  15]
 [  1 239]
 [  3 239]
 

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'execution' : {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-01-15 08:07:33', stop='2026-01-15 08:07:36', size=4096>)])},
'version' : 2,

The metadata of the PubResult result is:
'circuit_metadata' : {},


#### Wie der Estimator den Fehler berechnet
Zusätzlich zur Schätzung des Mittelwerts der in den Eingabe-PUBs übergebenen Observablen (das Feld `evs` des `DataBin`) versucht der Estimator auch, eine Schätzung des mit diesen Erwartungswerten verbundenen Fehlers zu liefern. Alle Estimator-Abfragen füllen das Feld `stds` mit einer Größe wie dem Standardfehler des Mittelwerts für jeden Erwartungswert, aber einige Fehlerminderungsoptionen liefern zusätzliche Informationen, wie zum Beispiel `ensemble_standard_error`.

Betrachte ein einzelnes Observable $\mathcal{O}$. In Abwesenheit von [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) kann man sich jeden Shot der Estimator-Ausführung als Punktschätzung des Erwartungswerts $\langle \mathcal{O} \rangle$ vorstellen. Wenn die punktweisen Schätzwerte in einem Vektor `Os` vorliegen, dann entspricht der in `ensemble_standard_error` zurückgegebene Wert dem Folgenden (wobei $\sigma_{\mathcal{O}}$ die [Standardabweichung der Erwartungswertsschätzung](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) und $N_{shots}$ die Anzahl der Shots ist):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

was alle Shots als Teil eines einzigen Ensembles behandelt. Wenn du Gate-[Twirling](/guides/error-mitigation-and-suppression-techniques#pauli-twirling) angefordert hast (`twirling.enable_gates = True`), kannst du die punktweisen Schätzwerte von $\langle \mathcal{O} \rangle$ in Mengen sortieren, die ein gemeinsames Twirl teilen. Nenne diese Schätzwert-Mengen `O_twirls`; es gibt `num_randomizations` (Anzahl der Twirls) davon. Dann ist `stds` der Standardfehler des Mittelwerts von `O_twirls`, wie in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

wobei $\sigma_{\mathcal{O}}$ die Standardabweichung von `O_twirls` und $N_{twirls}$ die Anzahl der Twirls ist. Wenn du Twirling nicht aktivierst, sind `stds` und `ensemble_standard_error` gleich.

Wenn du ZNE aktivierst, werden die oben beschriebenen `stds` zu Gewichten in einer nichtlinearen Regression an ein Extrapolationsmodell. Was letztendlich in `stds` zurückgegeben wird, ist die Unsicherheit des Fit-Modells, ausgewertet bei einem Rauschfaktor von null. Bei einem schlechten Fit oder einer großen Unsicherheit im Fit können die gemeldeten `stds` sehr groß werden. Wenn ZNE aktiviert ist, werden auch `pub_result.data.evs_noise_factors` und `pub_result.data.stds_noise_factors` befüllt, sodass du deine eigene Extrapolation durchführen kannst.
### Sampler-Ausgabe
Wenn ein Sampler-Job erfolgreich abgeschlossen wurde, enthält das zurückgegebene [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)-Objekt eine Liste von [`SamplerPubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.SamplerPubResult)s, eines pro PUB. Die DataBins dieser `SamplerPubResult`-Objekte sind wörterbuch-ähnliche Objekte, die ein `BitArray` pro `ClassicalRegister` im Circuit enthalten.

Die `BitArray`-Klasse ist ein Container für geordnete Shot-Daten. Genauer gesagt speichert sie die gesampelten Bitstrings als Bytes in einem zweidimensionalen Array. Die äußerste linke Achse dieses Arrays läuft über die geordneten Shots, während die äußerste rechte Achse über die Bytes läuft.

Als erstes Beispiel betrachten wir den folgenden Zehn-Qubit-Circuit: